In [ ]:
import pandas as pd
import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np 
from sklearn.manifold import TSNE
from sklearn import cluster, datasets, mixture
import matplotlib
import cmasher as cmr
from mycolorpy import colorlist as mcp
from scipy import stats

In [ ]:
import sklearn.preprocessing
import sklearn.pipeline
import scipy.spatial
import numpy as np


def calculate_hull(
        X, 
        scale=1.1, 
        padding="scale", 
        n_interpolate=100, 
        interpolation="quadratic_periodic", 
        return_hull_points=False):
    """
    Calculates a "smooth" hull around given points in `X`.
    The different settings have different drawbacks but the given defaults work reasonably well.
    Parameters
    ----------
    X : np.ndarray
        2d-array with 2 columns and `n` rows
    scale : float, optional
        padding strength, by default 1.1
    padding : str, optional
        padding mode, by default "scale"
    n_interpolate : int, optional
        number of interpolation points, by default 100
    interpolation : str or callable(ix,iy,x), optional
        interpolation mode, by default "quadratic_periodic"

    Inspired by: https://stackoverflow.com/a/17557853/991496
    """
    
    if padding == "scale":

        # scaling based padding
        scaler = sklearn.pipeline.make_pipeline(
            sklearn.preprocessing.StandardScaler(with_std=False),
            sklearn.preprocessing.MinMaxScaler(feature_range=(-1,1)))
        points_scaled = scaler.fit_transform(X) * scale
        hull_scaled = scipy.spatial.ConvexHull(points_scaled, incremental=True)
        hull_points_scaled = points_scaled[hull_scaled.vertices]
        hull_points = scaler.inverse_transform(hull_points_scaled)
        hull_points = np.concatenate([hull_points, hull_points[:1]])
    
    elif padding == "extend" or isinstance(padding, (float, int)):
        # extension based padding
        # TODO: remove?
        if padding == "extend":
            add = (scale - 1) * np.max([
                X[:,0].max() - X[:,0].min(), 
                X[:,1].max() - X[:,1].min()])
        else:
            add = padding
        points_added = np.concatenate([
            X + [0,add], 
            X - [0,add], 
            X + [add, 0], 
            X - [add, 0]])
        hull = scipy.spatial.ConvexHull(points_added)
        hull_points = points_added[hull.vertices]
        hull_points = np.concatenate([hull_points, hull_points[:1]])
    else:
        raise ValueError(f"Unknown padding mode: {padding}")
    
    # number of interpolated points
    nt = np.linspace(0, 1, n_interpolate)
    
    x, y = hull_points[:,0], hull_points[:,1]
    
    # ensures the same spacing of points between all hull points
    t = np.zeros(x.shape)
    t[1:] = np.sqrt((x[1:] - x[:-1])**2 + (y[1:] - y[:-1])**2)
    t = np.cumsum(t)
    t /= t[-1]

    # interpolation types
    if interpolation is None or interpolation == "linear":
        x2 = scipy.interpolate.interp1d(t, x, kind="linear")(nt)
        y2 = scipy.interpolate.interp1d(t, y, kind="linear")(nt)
    elif interpolation == "quadratic":
        x2 = scipy.interpolate.interp1d(t, x, kind="quadratic")(nt)
        y2 = scipy.interpolate.interp1d(t, y, kind="quadratic")(nt)

    elif interpolation == "quadratic_periodic":
        x2 = scipy.interpolate.splev(nt, scipy.interpolate.splrep(t, x, per=True, k=4))
        y2 = scipy.interpolate.splev(nt, scipy.interpolate.splrep(t, y, per=True, k=4))
    
    elif interpolation == "cubic":
        x2 = scipy.interpolate.CubicSpline(t, x, bc_type="periodic")(nt)
        y2 = scipy.interpolate.CubicSpline(t, y, bc_type="periodic")(nt)
    else:
        x2 = interpolation(t, x, nt)
        y2 = interpolation(t, y, nt)
    
    X_hull = np.concatenate([x2.reshape(-1,1), y2.reshape(-1,1)], axis=1)
    if return_hull_points:
        return X_hull, hull_points
    else:
        return X_hull


def draw_hull(
        X, 
        scale=1.1, 
        padding="scale", 
        n_interpolate=100, 
        interpolation="quadratic_periodic",
        plot_kwargs=None, 
        ax=None):
    """Uses `calculate_hull` to draw a hull around given points.

    Parameters
    ----------
    X : np.ndarray
        2d-array with 2 columns and `n` rows
    scale : float, optional
        padding strength, by default 1.1
    padding : str, optional
        padding mode, by default "scale"
    n_interpolate : int, optional
        number of interpolation points, by default 100
    interpolation : str or callable(ix,iy,x), optional
        interpolation mode, by default "quadratic_periodic"
    plot_kwargs : dict, optional
        `matplotlib.pyplot.plot` kwargs, by default None
    ax : `matplotlib.axes.Axes`, optional
        [description], by default None
    """

    if plot_kwargs is None:
        plot_kwargs = {}

    X_hull = calculate_hull(
        X, scale=scale, padding=padding, n_interpolate=n_interpolate, interpolation=interpolation)
    if ax is None:
        ax= plt.gca()
    plt.plot(X_hull[:,0], X_hull[:,1], **plot_kwargs)


def draw_rounded_hull(X, padding=0.1, line_kwargs=None, ax=None):
    """Plots a convex hull around points with rounded corners and a given padding.

    Parameters
    ----------
    X : np.array
        2d array with two columns and n rows
    padding : float, optional
        padding between hull and points, by default 0.1
    line_kwargs : dict, optional
        line kwargs (used for `matplotlib.pyplot.plot` and `matplotlib.patches.Arc`), by default None
    ax : matplotlib.axes.Axes, optional
        axes to plat on, by default None
    """

    default_line_kwargs = dict(
        color="black",
        linewidth=1
    )
    if line_kwargs is None:
        line_kwargs = default_line_kwargs
    else:
        line_kwargs = {**default_line_kwargs, **line_kwargs}

    if ax is None:
        ax = plt.gca()

    hull = scipy.spatial.ConvexHull(X)
    hull_points = X[hull.vertices]

    hull_points = np.concatenate([hull_points[[-1]], hull_points, hull_points[[0]]])

    diameter = padding * 2
    for i in range(1, hull_points.shape[0] - 1):

        # line
        
        # source: https://stackoverflow.com/a/1243676/991496
        
        norm_next = np.flip(hull_points[i] - hull_points[i + 1]) * [-1, 1]
        norm_next /= np.linalg.norm(norm_next)

        norm_prev = np.flip(hull_points[i - 1] - hull_points[i]) * [-1, 1]
        norm_prev /= np.linalg.norm(norm_prev)

        # plot line
        line = hull_points[i:i+2] + norm_next * diameter / 2
        ax.plot(line[:,0], line[:,1], **line_kwargs) 

        # arc

        angle_next = np.rad2deg(np.arccos(np.dot(norm_next, [1,0])))
        if norm_next[1] < 0:
            angle_next = 360 - angle_next

        angle_prev = np.rad2deg(np.arccos(np.dot(norm_prev, [1,0])))
        if norm_prev[1] < 0:
            angle_prev = 360 - angle_prev

        arc = patches.Arc(
            hull_points[i], 
            diameter, diameter,
            angle=0, fill=False, theta1=angle_prev, theta2=angle_next,
            **line_kwargs)

        ax.add_patch(arc)

In [ ]:
frac_plot_df = pd.read_pickle('HPA_anaysis/frac_plot_df.pkl')

In [ ]:
xx = pd.read_pickle('HPA_frac_df.pkl')

In [ ]:
raw = pd.read_excel('/Users/aelazar/Downloads/aal3321_thul_sm_table_s12.xlsx',sheet_name='hyperLOPIT Quantitation Data',header=1)
raw = raw.rename(columns={"Unnamed: 0": "acc"})

In [ ]:
AF2 = pd.read_pickle('../files/AF2_with_ENSG.pkl')
dSAM = pd.read_pickle('../files/dSAM_MPs_v_paper_v2_new_locs.pkl')
hTMs = pd.read_pickle('../files/hTMs_uniprot_v1.pkl')
markers = pd.read_excel('/Users/aelazar/Downloads/elife-16950-supp1-v3.xlsx',sheet_name='Organellar Markers HeLa')

In [ ]:
xx.to_excel('HPA_frac_dataset.xlsx')

In [ ]:
xx.mdSAM.value_counts()

In [ ]:
set.intersection(set.intersection(set(markers[markers.Compartment=='Lysosome']['Protein ID (canonical)']),set((xx[xx.gmm_new==2].index))),set(dSAM.acc))

In [ ]:
fraction = ['126','127C','127N','128C','128N','129C','129N','130C','130N','131']

In [ ]:

l = []
for acc in tqdm.tqdm(set(raw.acc)):
    for i in fraction:
        m = raw[raw.acc==acc].filter(regex=i).filter(regex='Replicate1').dropna(axis=1).values.mean()
        l.append({'acc':acc,'fraction':i,'val':m,'Replicate':1})
        m = raw[raw.acc==acc].filter(regex=i).filter(regex='Replicate2').dropna(axis=1).values.mean()
        
        l.append({'acc':acc,'fraction':i,'val':m,'Replicate':2})
hyperLOPIT = pd.DataFrame(l)



In [ ]:
for i in tqdm.tqdm(hyperLOPIT.index):
    if hyperLOPIT.loc[i,'acc'] in set(hTMs.acc):
        hyperLOPIT.loc[i,'tm'] = 'yes'
    else:
        hyperLOPIT.loc[i,'tm'] = 'no'
for i in tqdm.tqdm(hyperLOPIT.index):
    if hyperLOPIT.loc[i,'acc'] in set(dSAM.acc):
        hyperLOPIT.loc[i,'mdSAM'] = 'yes'
    else:
        hyperLOPIT.loc[i,'mdSAM'] = 'no'

In [ ]:
u= hyperLOPIT[(hyperLOPIT.tm=='yes')]
for i in tqdm.tqdm(u.index):
    acc= u.loc[i,'acc']
    if xx[xx.index==acc].empty ==False:
        u.loc[i,'gmm'] = xx[xx.index==acc].gmm_new.values[0]
cc = []#pd.DataFrame(columns=['x','y'])
for acc in tqdm.tqdm(set(u.acc)):
    gmm = u[u.acc==acc].gmm.values[0]
    for frac in fraction:
        #for r in [1,2]:
        v = u[(u.acc==acc)&(u.fraction==frac)&(u.Replicate==1)].val.values[0]
        vv = u[(u.acc==acc)&(u.fraction==frac)&(u.Replicate==2)].val.values[0]
        cc.append({'x':v,'y':vv,'frac':frac,'acc':acc,'gmm':gmm})
c3 = pd.DataFrame(cc)

In [ ]:
import matplotlib.colors as mcolors
cmap_test = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#D0D3D6",'#924E81E8',"#C03830","#6AB6B7","#94C47D","#F89B35",'#3A68AE'],10)
[mcolors.rgb2hex(cmap_test(i)) for i in range(10)]

In [ ]:
fig, ax = plt.subplots(figsize=(2,2))
sns.scatterplot(x='x',y='y',data=c3[c3.frac=='127C'],c='k',edgecolor=None,alpha=1,rasterized=True,hue='frac',palette='tab10')
plt.legend(ncol=2)
plt.locator_params(axis='x',nbins=1)
plt.locator_params(axis='y',nbins=1)
#stats.pearsonr(c3[c3.frac=='127C']['x'],c3[c3.frac=='127C']['y'])[0]
plt.xlim([0,.45])
plt.ylim([0,.45])
#g= sns.JointGrid(x='x',y='y',data=c3,height=4)#,edgecolor=None,s=10
#g.ax_joint.grid()
#g.plot_joint(sns.scatterplot,hue='frac',data=c3,edgecolor=None,s=25)
#g.plot_marginals(sns.violinplot,hue='ele',data=pd.DataFrame(precent_clusterd_list),split=True,saturation=1,fill=0,hue_order=['dAPP','APP'],palette=['gray','r'])


In [ ]:
fig = plt.figure(figsize = (3,9),constrained_layout=False)
axs = fig.subplots(nrows=5, ncols=2,gridspec_kw = {'hspace':.4,'wspace':.3})
c = 0
r2_list = []
for fra in fraction:
    r2 = stats.pearsonr(c3[c3.frac==fra]['x'],c3[c3.frac==fra]['y'])[0]
    sns.scatterplot(x='x',y='y',data=c3[c3.frac==fra],ax=axs.flat[c],c='k',edgecolor=None,alpha=.1,rasterized=True)
    axs.flat[c].set_title('%s r=%.2f'%(fra,r2),fontsize=16)
    axs.flat[c].locator_params(nbins=3)
    axs.flat[c].grid()
    axs.flat[c].set_xlim([-.1,c3[c3.frac==fra].loc[:,['x','y']].max().max()+.1])
    axs.flat[c].set_ylim([-.1,c3[c3.frac==fra].loc[:,['x','y']].max().max()+.1])
    r2_list.append({'frac':fra,'val':r2})
    if c==5:
        axs.flat[c].set_xlabel('Replicate 1',fontsize=16)
        axs.flat[c].set_ylabel('Replicate 2',fontsize=16)
    else:
        axs.flat[c].set_xlabel('',fontsize=16)
        axs.flat[c].set_ylabel('',fontsize=16)

    c = c+1
plt.savefig('hyperLOPIT_TM_corr_v2.svg',bbox_inches='tight',transparent=True)

In [ ]:
from scipy import stats
fig, ax = plt.subplots(figsize=(5,5)) 
sns.scatterplot(x='x',y='y',data=c3,hue='frac',alpha=.1,palette=[mcolors.rgb2hex(cmap_test(i)) for i in range(10)],edgecolor=None,rasterized=True)#
print(stats.pearsonr(c3['x'],c3['y']))
sns.lineplot(x=[0,.4], y=[0,.4], color='k',ls='--')
plt.xlabel('$Channel$\n$_{(Replicate\;1)}$',fontsize=12)
plt.ylabel('$Channel$\n$_{(Replicate\;2)}$',fontsize=12)
plt.text(.05,.35,'$r=%.2f$'%stats.pearsonr(c3.x,c3.y)[0])
plt.legend(fontsize='xx-small')
ax.grid()
plt.xlim([-.01,.4])
plt.ylim([-.01,.4])
plt.locator_params(axis='y', nbins=4)
plt.locator_params(axis='x', nbins=4)
plt.legend(title='Channel',fontsize=12,title_fontsize=8)
plt.savefig('hyperLOPIT_TM_corr.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig, ax = plt.subplots(figsize=(2,4))
sns.barplot(y='frac',x='val',data=pd.DataFrame(r2_list),width=.98,color='gray')
#ax.grid()
plt.xlim([0.65,.95])
plt.locator_params(nbins=4)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
plt.xlabel('')
plt.ylabel('')
plt.savefig('hyperLOPIT_TM_corr_bar.svg',bbox_inches='tight',transparent=True)

In [ ]:
hyperLOPIT

In [ ]:
for i in tqdm.tqdm(hyperLOPIT.index):
    if markers[markers['Protein ID (canonical)']==hyperLOPIT.loc[i,'acc']].empty ==False:
        hyperLOPIT.loc[i,'Compartment'] = markers[markers['Protein ID (canonical)']==hyperLOPIT.loc[i,'acc']].Compartment.values[0]

In [ ]:
['ER','Endosome','Golgi','Plasma membrane','Lysosome','Mitochondrion','Nuclear pore complex']

In [ ]:
fig = plt.figure(figsize = ((4),4))#20,10
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
axs = fig.subplots(nrows=2, ncols=2,gridspec_kw = {'hspace':.3,'wspace':.3})
axs.ravel()

for i,j,c in zip([0,1,2,3],['PM','ER','Endo/Lyso','Golgi'],['#95AAD3','#23A993','#EDAEAE','#E0A458']):
    sns.lineplot(x='frac',y='val',
                 data=frac_plot_df[frac_plot_df.gmm_new==i],
                 units='acc',estimator=None,alpha=.1,c=c,ax=axs.ravel()[i],rasterized=True)
    # sns.lineplot(x='frac',y='val',
    #              data=frac_plot_df[frac_plot_df.gmm_new==1],
    #              estimator=np.mean,c='k',errorbar=None)
    axs.ravel()[i].grid()
    axs.ravel()[i].set_ylim([0,.4])
    axs.ravel()[i].text(7,.35,'$%d$'%len(set(frac_plot_df[frac_plot_df.gmm_new==i].acc)),fontsize=8)
    axs.ravel()[i].set_title('$%s$'%j)
    if i==2:
        axs.ravel()[i].set_xlabel('$Channels$')
        axs.ravel()[i].set_ylabel('$Intensity$')
        axs.ravel()[i].set_xticklabels(labels=fraction,rotation=45,fontsize=8)
    else:
        axs.ravel()[i].set_xlabel('')
        axs.ravel()[i].set_ylabel('')
        axs.ravel()[i].set_xticklabels([])
        axs.ravel()[i].set_yticklabels([])
    plt.xticks(rotation=45)
#plt.savefig('HPA_anaysis/hyperLOPIT_compartments.svg',bbox_inches='tight',transparent=True)

In [ ]:
fraction

In [ ]:
frac_plot_df['category'] = pd.Categorical(frac_plot_df['frac'], categories=fraction, ordered=True)
fig, ax = plt.subplots(figsize=(2.75,2.75)) 
sns.lineplot(x='category',y='val',
             data=frac_plot_df[frac_plot_df.mdSAM=='-'],
             units='acc',estimator=None,alpha=.1,c='gray',rasterized=False)

In [ ]:
fig, ax = plt.subplots(figsize=(2.75,2.75)) 
sns.lineplot(x='category',y='val_100',
             data=frac_plot_df[frac_plot_df.mdSAM=='-'],
             units='acc',estimator=None,alpha=.1,c='gray',rasterized=False)

sns.lineplot(x='category',y='val_100',
             data=frac_plot_df[frac_plot_df.mdSAM=='+'],
             units='acc',estimator=None,alpha=.1,c='#C03830',rasterized=False)
#plt.ylim([0,4])
plt.ylabel('')
plt.xlabel('')
#plt.ylim([0,.45])
#ax.grid()
plt.legend([],frameon=False)
plt.xticks(rotation=45)
#plt.text(5.5,.4,'$n=900\,MPs$',fontsize=10)
#plt.text(1,.35,'$Membrane$',fontsize=10)
#plt.legend(['+','-'],title='$mdSAM$',fontsize=10,loc='upper left')
#plt.title('$hyperLOPIT$\n$_{U2OS\;cells}$')
plt.savefig('HPA_anaysis/hyperLOPIT_raw.svg',bbox_inches='tight',transparent=True)

In [ ]:
frac_plot=[]
for acc in xx.index:
    for f in fraction:
        frac_plot.append({'acc':acc,'frac':f,'val':xx.loc[acc,f],'mdSAM':xx.loc[acc,'mdSAM'],
                          'Compartment':xx.loc[acc,'Compartment'],
                          'gmm_new':xx.loc[acc,'gmm_new'],'idr':xx.loc[acc,'idr']})

In [ ]:
frac_plot_df = pd.DataFrame(frac_plot)

In [ ]:
hyperLOPIT[hyperLOPIT.acc.isin(frac_plot_df.acc)].groupby(['fraction','acc']).mean().reset_index().pivot_table(values='val',
    index='acc',
    columns='fraction')*100

In [ ]:
g = sns.clustermap(hyperLOPIT[hyperLOPIT.acc.isin(frac_plot_df.acc)].groupby(['fraction','acc']).mean().reset_index().pivot_table(values='val',
    index='acc',
    columns='fraction')*100,cmap='cividis',dendrogram_ratio=0.1,figsize=(3,4),col_cluster=0,rasterized=True)
g.ax_heatmap.set_yticklabels([])
plt.savefig('hyperLOPIT_clustermap.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig, ax = plt.subplots(figsize=(2.75*1.618,2.75)) 
sns.lineplot(x='frac',y='val',lw=1,ls='-',
             data=frac_plot_df,estimator=None,units='acc',alpha=.1,
             palette=['#C03830','#808080'],hue_order=['+','-'],hue='mdSAM',marker="o",markersize=0,markeredgecolor=None)
# sns.scatterplot(x='fraction',y='val',data=pd.DataFrame(frac_plot),alpha=.1,s=20,edgecolor=None,
#              )
plt.ylabel('$Intensity$')
plt.xlabel('$Channels$')
plt.ylim([0,.45])
ax.grid()
plt.legend([],frameon=False)
plt.xticks(rotation=45)
plt.text(7,.35,'$n=900\,MPs$',fontsize=10)
#plt.text(1,.35,'$Membrane$',fontsize=10)
plt.title('$U2OS\;hyperLOPIT$\n$_{Thul\;et\;al.}$')
plt.tight_layout()
#plt.savefig('HPA_anaysis/hyperLOPIT_raw.svg',bbox_inches='tight',transparent=True)

In [ ]:
frac_plot_df['idr_log'] = np.log10(frac_plot_df['idr'])

In [ ]:
for i in frac_plot_df.index:
    frac_plot_df.loc[i,'cluster_int'] = '%d'%int(frac_plot_df.loc[i,'gmm_new'])

In [ ]:
fig, ax = plt.subplots(figsize=(2*1.618,2)) 
sns.lineplot(x='frac',y='val',lw=1,ls='-',
             data=frac_plot_df[frac_plot_df.gmm_new==0],estimator=None,units='acc',alpha=.1
             ,c='#3A68AE',marker="o",markersize=0,markeredgecolor=None)
# sns.scatterplot(x='fraction',y='val',data=pd.DataFrame(frac_plot),alpha=.1,s=20,edgecolor=None,
#              )
plt.ylabel('$Intensity$')
# plt.ylim([0,.45])
ax.grid()
plt.legend([],frameon=False)
plt.xticks(rotation=45)
plt.text(7.5,.25,'$n=244$',fontsize=10)
plt.title('$0$')
plt.tight_layout()

In [ ]:
hyperLOPIT = hyperLOPIT[~hyperLOPIT.acc.isin(delete)]#.Compartment.drop_duplicates()

In [ ]:
gmm

In [ ]:
xx = hyperLOPIT[(hyperLOPIT.tm=='yes')].pivot_table(index='acc',columns='fraction',values='val')
gmm = mixture.GaussianMixture(n_components=8, covariance_type="tied")
principalComponents = TSNE(n_components=2, learning_rate='auto',init='random', perplexity=10).fit_transform(xx)
principalDf = pd.DataFrame(data = principalComponents
             , columns = ['x', 'y'])
#embedding = reducer.fit_transform(xx)
xx['x']='A'
xx['y']='A'
xx['mdSAM']='A'
for i in xx.index:
    if AF2[AF2.acc==i].pLDDT.empty==False:
        if AF2[AF2.acc==i].pLDDT.values[0]!='A':
            le = len([i for i in AF2[AF2.acc==i].pLDDT.values[0] if i<70])
            xx.loc[i,'idr']=le
for i in enumerate(embedding):
    xx.at[xx.index[i[0]],'x'] = principalDf.loc[i[0],'x']#embedding[i[0]][0]#principalDf.loc[i[0],'x']##principalDf.loc[i[0],'x']#
    xx.at[xx.index[i[0]],'y'] = principalDf.loc[i[0],'y']#embedding[i[0]][1]#principalDf.loc[i[0],'y']#embedding[i[0]][1]#principalDf.loc[i[0],'y']#embedding[i[0]][1]
    if xx.index[i[0]] in set(dSAM.acc):
        xx.at[xx.index[i[0]],'mdSAM'] = '+'
    else:
        xx.at[xx.index[i[0]],'mdSAM'] = '-'
for i in xx.index:
    xx.loc[i,'acc'] = i
    if markers[markers['Protein ID (canonical)']==i].empty ==False:
        xx.loc[i,'Compartment'] = markers[markers['Protein ID (canonical)']==i].Compartment.values[0]
gmm.fit(xx.loc[:,['x','y']])
#xx = hyperLOPIT[(hyperLOPIT.tm=='yes')].pivot_table(index='acc',columns='fraction',values='val')


for i in enumerate(gmm.predict(xx.loc[:,['x','y']])):
    xx.at[xx.index[i[0]],'gmm'] = i[1]

In [ ]:
xx

In [ ]:
d = {2:0,6:0,0:1,5:1,7:1,1:2,3:3,4:4}
for i in enumerate(xx.index):
    xx.loc[i[1],'gmm_new']=(d[xx.loc[i[1],'gmm']])

In [ ]:
ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#18827E","gray","#B34F91"])#"#"
colors = cmr.take_cmap_colors(ww, 5, return_fmt='hex')


In [ ]:
colors = [mcolors.rgb2hex(ww(i)) for i in range(5)]

In [ ]:
#pd.Series(xx[(xx.gmm_new==4)&(xx.mdSAM=='+')].index).to_clipboard(index=0)

In [ ]:
fig, ax = plt.subplots(figsize=(3,3*1.618)) 
cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", ['#3A68AE',"#F89B35",'#D8AEC8',"#6AB6B7","#94C47D"]) #["#3A68AE",'#C2AFF0',"#09BC8A","#F28F3B","#726E60"]["#3A68AE",'#925082B2',"#434E6C"]##FF9B00","#2E4185"]
colors = pd.Series(colors).sample(5).to_list()
#colors =  ['#95AAD3','#23A993','#EDAEAE','#E0A458','#7E7CCB'] #['#3A68AE',"#F89B35",'#D8AEC8',"#6AB6B7","#94C47D"]
ax  = sns.scatterplot(x='x',y='y',data=xx,hue='gmm_new',s=30,edgecolor=None,palette=colors,legend=[int(i) for i in set(xx.gmm_new)])

for j in range(5):
    sns.scatterplot(x='x',y='y',data=xx[(xx.mdSAM=='+')&(xx.gmm_new==j)],s=30,edgecolor='k',color=colors[j],alpha=1)##C03830

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [int(i) for i in range(5)],ncol=1,fontsize=7,loc='lower left')
plt.xlabel('$t‑SNE$')
ax.grid()
plt.title('$U2OS\;hyperLOPIT$')
plt.ylabel('$t‑SNE$')
plt.text(-0,30,'$PM$')
plt.text(30,-55,'$ER$')
plt.text(-65,58,'$Endo/Lyso$')
plt.text(16,50,'$Golgi$')
plt.text(-40,-50,'$Other$')
plt.ylim([-65,65])
plt.xlim([-75,75])
plt.locator_params(axis='y', nbins=4)
plt.locator_params(axis='x', nbins=4)
plt.text(55,-40,'$SSR1$\n$RTN1$\n$STX5$\n$MIA3$\n$SARAF$',fontsize=7)
plt.text(-70,-50,'$OXA1L$\n$RRP12$\n$INTS5$\n$MUC$\n$FUNDC2$',fontsize=7)      
plt.text(-60,-10,'$ADGRB1$\n$APP$\n$NGFR$\n$MET$\n$CDH2$',fontsize=7)    
plt.text(42,32,'$GOLIN4$\n$MAN1A2$\n$QSOX2$\n$GLG1$\n$MAN1B1$',fontsize=7)
plt.text(-70,40,'$ATG9A$\n$SORT1$\n$ATP13A3$',fontsize=7)
plt.legend(ncol=5)
#plt.savefig('HPA_anaysis/hyperLOPIT_tsne_v4.svg',bbox_inches='tight',transparent=True)

In [ ]:
xx

In [ ]:
fig, ax = plt.subplots(figsize=(3,4)) 
ax.grid()
cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", ['#3A68AE',"#F89B35",'#D8AEC8',"#6AB6B7","#94C47D"]) #["#3A68AE",'#C2AFF0',"#09BC8A","#F28F3B","#726E60"]["#3A68AE",'#925082B2',"#434E6C"]##FF9B00","#2E4185"]
colors = pd.Series(colors).sample(5).to_list()
#colors =  ['#95AAD3','#23A993','#EDAEAE','#E0A458','#7E7CCB'] #['#3A68AE',"#F89B35",'#D8AEC8',"#6AB6B7","#94C47D"]

for i,j in zip([0,1,2,3,4], ['#C6C6C8','#5A75A3','#76A5CB','#8EB6D5','#BB5450']):#,'#E73490'['#674CAC','#BB1919','#3C4776']
    #hull = ConvexHull(df_membrane[df_membrane.gmm_dSAMs==i].loc[:,['x','y']].to_numpy())
    points =xx[xx.gmm_new==i].loc[:,['x','y']].to_numpy()
    draw_hull(points,scale=1,n_interpolate=40,plot_kwargs={'color':j,'lw':2,'zorder':5})


ax  = sns.scatterplot(x='x',y='y',data=xx,s=20,edgecolor=None,color='#808080')

for j in range(5):
    sns.scatterplot(x='x',y='y',data=xx[(xx.mdSAM=='+')&(xx.gmm_new==j)],s=20,edgecolor=None,color='#C03830',alpha=1)##C03830

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [int(i) for i in range(5)],ncol=1,fontsize=7,loc='lower left')
#plt.xlabel('$t‑SNE$')

#plt.title('$U2OS\;hyperLOPIT$')
#plt.ylabel('$t‑SNE')
# plt.text(-0,30,'$PM$')
# plt.text(30,-55,'$ER$')
# plt.text(-65,58,'$Endo/Lyso$')
# plt.text(16,50,'$Golgi$')
# plt.text(-40,-50,'$Other$')
plt.ylim([-65,65])
plt.xlim([-75,75])
plt.locator_params(axis='y', nbins=4)
plt.locator_params(axis='x', nbins=4)
ax.axis('off')
plt.legend([],frameon=False)
# plt.text(55,-40,'$SSR1$\n$RTN1$\n$STX5$\n$MIA3$\n$SARAF$',fontsize=7)
# plt.text(-70,-50,'$OXA1L$\n$RRP12$\n$INTS5$\n$MUC$\n$FUNDC2$',fontsize=7)      
# plt.text(-60,-10,'$ADGRB1$\n$APP$\n$NGFR$\n$MET$\n$CDH2$',fontsize=7)    
# plt.text(42,32,'$GOLIN4$\n$MAN1A2$\n$QSOX2$\n$GLG1$\n$MAN1B1$',fontsize=7)
# plt.text(-70,40,'$ATG9A$\n$SORT1$\n$ATP13A3$',fontsize=7)
#plt.legend(ncol=5)
plt.savefig('HPA_anaysis/hyperLOPIT_tsne_v5.svg',bbox_inches='tight',transparent=True)

In [ ]:
len(xx[(xx.mdSAM=='+')&(xx.gmm_new==0)])

In [ ]:
len(xx[(xx.gmm_new==0)&(xx.mdSAM=='+')])/len(xx[(xx.gmm_new==0)])

In [ ]:
xx.loc[xx[xx.Compartment=='Ergic/cisGolgi'].index,'Compartment'] = 'ERGIC'

In [ ]:
['$%s$'%i.replace('_','\;').replace(' ','\;') for i in labels[0:10]]

In [ ]:
[text.get_text() for text in ax.legend_.texts]

In [ ]:
aa_len = []
for i in AF2[AF2.acc.isin(hTM.acc)].index:
    if AF2.loc[i,'pLDDT']!='A':
        aa_len.append(np.log10(len(AF2.loc[i,'AAseq'])))

In [ ]:
((10**2.6)*110)/1000

In [ ]:
['$%s$'%i.replace('_','\,').replace(' ','\,') for i in set(xx.Compartment.dropna())]

In [ ]:
ll= []
for j in xx['gmm_new']:
    for i in set(xx.Compartment.dropna()):
        ll.append({'gmm_clster':int(j),'Compartment':i,'frac':100*len(xx[(xx['gmm_new']==j)&(xx.Compartment==i)])/len(xx[(xx.Compartment==i)])})
pd.DataFrame(ll)
sns.barplot(x='gmm_clster',y='frac',data=pd.DataFrame(ll),hue='Compartment',dodge=True)
plt.legend(fontsize='x-small') 

In [ ]:
xx = pd.read_pickle('HPA_frac_df.pkl')

In [ ]:
heatmap = pd.DataFrame(ll).pivot_table(index='Compartment',columns='gmm_clster',values='frac')

In [ ]:
heatmap_order = heatmap.loc[['Plasma membrane','ER', 'ERGIC', 'ER_high_curvature', 'Endosome', 'Lysosome', 'Golgi',
       'Mitochondrion', 'Nuclear pore complex', 'Peroxisome'],:
            ]
text_heat = ['%s'%i.replace('_',' ').replace(' ',' ') for i in ['Plasma membrane','ER', 'ERGIC', 'ER_high_curvature', 'Endosome', 'Lysosome', 'Golgi',
       'Mitochondrion', 'Nuclear pore complex', 'Peroxisome']]

In [ ]:
heatmap_order

In [ ]:
print(xx.mdSAM.value_counts())

In [ ]:
len(xx[(xx.gmm_new==3)&(xx.mdSAM=='+')])

In [ ]:
ll= []
for j in xx['gmm_new']:
    for i in set(xx.Compartment.dropna()):
        ll.append({'gmm_clster':int(j),'Compartment':i,'frac':len(xx[(xx['gmm_new']==j)&(xx.Compartment==i)])})
pd.DataFrame(ll)
heatmap_annot = pd.DataFrame(ll).pivot_table(index='Compartment',columns='gmm_clster',values='frac')
# sns.barplot(x='gmm_clster',y='frac',data=pd.DataFrame(ll),hue='Compartment',dodge=True)
# plt.legend(fontsize='x-small') 

heatmap_annot = heatmap_annot.loc[['Plasma membrane','ER', 'ERGIC', 'ER_high_curvature', 'Endosome', 'Lysosome', 'Golgi',
       'Mitochondrion', 'Nuclear pore complex', 'Peroxisome'],:
            ]
heatmap_annot

In [ ]:
fig, ax = plt.subplots(figsize=(5*1.618,5)) 
sns.heatmap(heatmap_order,square=True,
          cmap='binary',ax=ax,lw=0,cbar_kws={"shrink": 0.5,'label': '%',"ticks":[0,50,100]},alpha=1, linecolor='black',annot=heatmap_annot)
ax.set_yticklabels(text_heat,fontsize=9)
ax.figure.axes[-1].yaxis.label.set_size(16)
cbar = ax.collections[0].colorbar
# here set the labelsize by 20
cbar.ax.tick_params(labelsize=16)
sns.despine(left=False, bottom=False,top=False,right=False)
plt.xlabel('GMM cluster',fontsize=16)
plt.ylabel('Organelle',fontsize=16)
ax.set_title('Organelle fiducial markers',fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(),fontsize=16,rotation=0)
plt.savefig('HPA_anaysis/hyperLOPIT_tsne_2_matrix.svg',bbox_inches='tight',transparent=True)

In [ ]:
#print(xx[xx.gmm==2].mdSAM.value_counts())
#print(xx.mdSAM.value_counts())
fisher = []
from scipy.stats.contingency import expected_freq
from scipy.stats import fisher_exact
def enrich(o,e):
    return np.log2(o/e)


for cluster in range(5):
    print(cluster)
    table = [741-len(xx[(xx.mdSAM=='-')&(xx.gmm_new==cluster)]),159-len(xx[(xx.mdSAM=='+')&(xx.gmm_new==cluster)])],[len(xx[(xx.mdSAM=='-')&(xx.gmm_new==cluster)]),len(xx[(xx.mdSAM=='+')&(xx.gmm_new==cluster)])]
    print(table)
    #print(cluster)
    #print(table)
    res = fisher_exact(table)
    #print(res)
    expected_table = expected_freq(table)
    fisher.append({'gmm_cluster':cluster,'gmm_cluster_size':len(xx[xx.gmm_new==cluster]),
                   'obs':table[-1][-1],'exp':expected_table[-1][-1],'pval':res[-1],'enrich':enrich(table[-1][-1],expected_table[-1][-1])})
    #print(table_sams)
    
    print(expected_table)
    print('---')
    #print()
    #print(enrich(table[-1][-1],expected_table[-1][-1]),res)

In [ ]:
fisher_bar = pd.DataFrame(fisher)
fisher_bar

In [ ]:
fisher_bar

In [ ]:
l= []
for i in fisher_bar.index:
    
    l.append({'gmm_cluster':fisher_bar.loc[i,'gmm_cluster'],'val':fisher_bar.loc[i,'obs'],'ele':'obs','pval':fisher_bar.loc[i,'pval']})
    l.append({'gmm_cluster':fisher_bar.loc[i,'gmm_cluster'],'val':fisher_bar.loc[i,'exp'],'ele':'exp','pval':fisher_bar.loc[i,'pval']})

In [ ]:
fisher_df = pd.DataFrame(l)
fisher_df.drop_duplicates(subset='pval')


In [ ]:
fig, ax = plt.subplots(figsize=(2.5*1.618,2.5)) 
sns.barplot(x='gmm_cluster',y='val',data=pd.DataFrame(l),hue='ele',hue_order=['obs','exp'],lw=0,palette=['#C03830','k'])

In [ ]:
pd.DataFrame(l)

In [ ]:

fig, ax = plt.subplots(figsize=(4,4)) 

bar = sns.barplot(x='gmm_cluster',y='val',data=pd.DataFrame(l),hue='ele',hue_order=['obs','exp'],lw=0,zorder=0,width=0.95)
ax.grid(zorder=0)
plt.ylabel('No. mdSAMs',fontsize=16)
plt.xlabel('')
# for i in enumerate(fisher_df.drop_duplicates(subset='pval').index):
#     if fisher_df.loc[i[1],'pval']>.05:
#         e='ns'
#         plt.text(i[0]-.3,5,'$ns$'
#                  ,rotation=0,fontsize=10,color='k')
#     else:
#         e = '%1.10e' % fisher_df.pval[i[1]]
#         lead = e[0]
#         power = e.split('-')[-1].split('0')[-1]
#         plt.text(i[0]-.3,5,'$p=%s⋅10^{‑%s}$'%(lead,power),rotation=90,fontsize=10,color='k')

plt.ylim([-0,80])
plt.xlim([-.75,4.75])
#plt.legend(['$Observed$','$Expected$'])
L=plt.legend(fontsize=10)
ax.locator_params(axis='y', nbins=6)
L.get_texts()[0].set_text('$Observed$')
L.get_texts()[1].set_text('$Expected$')
ax.set_xticklabels(['$PM$','$ER$','$Endo/Lyso$','$Golgi$','$Other$'],rotation=90,fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)

hatches = ['\\\\' for i in range(12)]

# Loop over the bars
c = 0
colors = ['#807F80','#23A993','#67A7B0','#B04E8F','#E3B167','#807F80','#23A993','#67A7B0','#B04E8F','#E3B167']
for i,thisbar in enumerate(bar.patches):
    # Set a different hatch for each bar
    if i in [5,6,7,8,9]:
        
        thisbar.set_facecolor(colors[c])
        thisbar.set_hatch(hatches[i])
        thisbar.set_edgecolor('white')
        c = c+1
c = 0
for i,thisbar in enumerate(bar.patches):
    # Set a different hatch for each bar
    if i not in [5,6,7,8,9]:
        thisbar.set_facecolor(colors[c])
        c = c+1
plt.rcParams['hatch.linewidth'] = 1.5
plt.savefig('HPA_anaysis/hyperLOPIT_enrich_v2.svg',bbox_inches='tight',transparent=True)

In [ ]:

fig, ax = plt.subplots(figsize=(4,4)) 

bar = sns.barplot(x='gmm_cluster',y='val',data=pd.DataFrame(l),hue='ele',hue_order=['obs','exp'],lw=0,zorder=0,width=0.95)
ax.grid(zorder=0)
plt.ylabel('No. mdSAMs',fontsize=16)
plt.xlabel('')
# for i in enumerate(fisher_df.drop_duplicates(subset='pval').index):
#     if fisher_df.loc[i[1],'pval']>.05:
#         e='ns'
#         plt.text(i[0]-.3,5,'$ns$'
#                  ,rotation=0,fontsize=10,color='k')
#     else:
#         e = '%1.10e' % fisher_df.pval[i[1]]
#         lead = e[0]
#         power = e.split('-')[-1].split('0')[-1]
#         plt.text(i[0]-.3,5,'$p=%s⋅10^{‑%s}$'%(lead,power),rotation=90,fontsize=10,color='k')

plt.ylim([-0,80])
plt.xlim([-.75,4.75])
#plt.legend(['$Observed$','$Expected$'])
L=plt.legend(fontsize=10)
ax.locator_params(axis='y', nbins=6)
L.get_texts()[0].set_text('$Observed$')
L.get_texts()[1].set_text('$Expected$')
ax.set_xticklabels(['$PM$','$ER$','$Endo/Lyso$','$Golgi$','$Other$'],rotation=90,fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)

hatches = ['\\\\' for i in range(12)]

# Loop over the bars
c = 0
colors = ['#C03830','#C03830','#C03830','#C03830','#C03830','#C03830','#C03830','#C03830','#C03830','#C03830']
for i,thisbar in enumerate(bar.patches):
    # Set a different hatch for each bar
    if i in [5,6,7,8,9]:
        
        thisbar.set_facecolor(colors[c])
        thisbar.set_hatch(hatches[i])
        thisbar.set_edgecolor('white')
        c = c+1
c = 0
for i,thisbar in enumerate(bar.patches):
    # Set a different hatch for each bar
    if i not in [5,6,7,8,9]:
        thisbar.set_facecolor(colors[c])
        c = c+1
plt.rcParams['hatch.linewidth'] = 1.5
plt.savefig('HPA_anaysis/hyperLOPIT_enrich_v2.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig, ax = plt.subplots(figsize=(4,4)) 

bar = sns.barplot(x='gmm_cluster',y='val',data=pd.DataFrame(l),hue='ele',hue_order=['obs','exp'],lw=0,zorder=0,width=0.95,order=[4,3,2,1,0],palette=['#C03830','#A4A5A5'])
ax.grid(zorder=0)
plt.ylabel('No. mdSAMs',fontsize=12)
plt.xlabel('')
# for i in enumerate(fisher_df.drop_duplicates(subset='pval').index):
#     if fisher_df.loc[i[1],'pval']>.05:
#         e='ns'
#         plt.text(i[0]-.3,5,'$ns$'
#                  ,rotation=0,fontsize=10,color='k')
#     else:
#         e = '%1.10e' % fisher_df.pval[i[1]]
#         lead = e[0]
#         power = e.split('-')[-1].split('0')[-1]
#         plt.text(i[0]-.3,5,'$p=%s⋅10^{‑%s}$'%(lead,power),rotation=90,fontsize=10,color='k')

plt.ylim([-0,80])
plt.xlim([-.75,4.75])
#plt.legend(['$Observed$','$Expected$'])
L=plt.legend(fontsize=10)
ax.locator_params(axis='y', nbins=6)
L.get_texts()[0].set_text('$Observed$')
L.get_texts()[1].set_text('$Expected$')
ax.set_xticklabels(['','','','',''],rotation=90,fontsize=16)


In [ ]:
df_localization  = pd.DataFrame(l)

In [ ]:
df_localization

In [ ]:
mapper = dict(zip([0,1,2,3,4],['PM','ER','Endo/Lyso','Golgi','Mito & Other']))
df_localization['loca'] = df_localization['gmm_cluster'].map(mapper)
l = []
for i in set(df_localization.gmm_cluster):
    obs = df_localization[(df_localization.gmm_cluster==i)&(df_localization.ele=='obs')].val.values[0]
    exp = df_localization[(df_localization.gmm_cluster==i)&(df_localization.ele!='obs')].val.values[0]
    orga = df_localization[(df_localization.gmm_cluster==i)&(df_localization.ele=='obs')].loca.values[0]
    enrich = (np.log2((obs/exp)))
    pval = df_localization[(df_localization.gmm_cluster==i)&(df_localization.ele=='obs')].pval.values[0]
    l.append({'orga':orga,'enrich':enrich,'pval':pval,'obs':obs})

In [ ]:
def fdr(p_vals):

    from scipy.stats import rankdata
    ranked_p_values = rankdata(p_vals)
    fdr = p_vals * len(p_vals) / ranked_p_values
    fdr[fdr > 1] = 1

    return fdr
df_orga = pd.DataFrame(l)
df_orga['fdr'] = fdr(df_orga['pval'])
df_orga['fdr_log']= -1*np.log10(df_orga['fdr'])

In [ ]:
fig, ax = plt.subplots(figsize=(3,4)) 
ax = sns.barplot(x='enrich',y='orga',data=df_orga.sort_values(by='enrich',ascending=0),lw=0,ax=ax,zorder=2)
ax.grid(zorder=0)#,inner_kws=dict(box_width=10))#,order=['Cytoplasmic','Extracellular','Lumenal'])
#ax.grid()
plt.locator_params(nbins=4)
plt.axvline(0,ls='--',color='k')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,fontsize=16)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0,fontsize=16)
# sns.histplot(y='obs',data=df_orga,ax=ax2
#                   ,hue='orga',multiple="stack",lw=.5,bins=1,alpha=1,
#                   edgecolor='w',zorder=10,)
# ax.set_ylabel('No. amino acids (to closest leaflet)',fontsize=14)
# ax.set_xticklabels(['CP','EC','Lumenal'],fontsize=14)
# ax.set_ylim([1,2e4])

# #plt.xlabel('')

# ax.yaxis.set_tick_params(labelsize = 12)
# ax.set_xlabel('mdSAM location',fontsize=16)
# sns.histplot(x='val',data=pd.DataFrame(l),ax=ax2
#                   ,hue='type',multiple="stack",lw=.5,bins=1,alpha=1,
#                   edgecolor='w',zorder=10,palette=['#8C8C8C','#043278','#397BDE'])
# ax2.set_xlabel('')
# ax.set_xticklabels([])
# ax.set_xlabel('')
# ax.set_ylabel('')
# ax2.set_ylabel('Count',fontsize=14)
# ax2.set_yticklabels(ax2.get_yticklabels(), rotation=0,fontsize=12)
# ax2.set_xticklabels([])
# ax2.locator_params(axis='y', nbins=6)
# ax2.legend([],frameon=False)
plt.savefig('mdSAM_orgnalle_v5.svg',bbox_inches='tight',transparent=True)


In [ ]:
weight_counts = {
'Mito & Other': 15.0,
'ER': 41.0,
'Golgi': 10.0,
'Endo/Lyso': 22.0,
'PM': 71.0}

In [ ]:
species = (
    " ")
#weight_counts= dict(zip(df_orga.orga,df_orga.obs))
width = .1
colors=['#BB5450','#5A75A3','#8EB6D5','#76A5CB','#C6C6C8']
fig, ax = plt.subplots(figsize=(1,4)) 
bottom = np.zeros(1)
c = 0
for boolean, weight_count in weight_counts.items():
    p = ax.bar(species, weight_count, width, label=boolean, bottom=bottom,color=colors[c])
    bottom += weight_count
    c = c+1
plt.ylabel('',fontsize=16)
plt.locator_params(nbins=5)
plt.yticks(fontsize=14, rotation=0)
#plt.savefig('donut_v3.svg',bbox_inches='tight',transparent=True)
#sns.despine(ax=ax)
plt.savefig('HPA_OBS_ORGA.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig, ax = plt.subplots(figsize=(4,4)) 
ax.grid(zorder=0)
sns.scatterplot(x='enrich',y='fdr_log',data=df_orga,s=100,zorder=1,color='k')
plt.xlabel('FC(log$_{2}$)',fontsize=12)
plt.ylabel('FDR(-log$_{10}$)',fontsize=12)
plt.axhline(-1*np.log10(.05),ls='--',color='k',zorder=0)
plt.axvline(.5,ls='--',color='k',zorder=0)
plt.axvline(-.5,ls='--',color='k',zorder=0)
plt.xlim([-1,1])
plt.locator_params(nbins=4)
plt.savefig('HPA_anaysis/hyperLOPIT_enrich_v3.svg',bbox_inches='tight',transparent=True)

In [ ]:
#### from matplotlib import colors
divnorm=colors.TwoSlopeNorm(vmin=-3, vcenter=0., vmax=.5)
ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#18827E","#B34F91"])#"#F3F3F3"
fig, ax = plt.subplots(figsize=(2,4)) 
sns.set(font_scale=.75,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})

sns.heatmap(pd.DataFrame(l).pivot(index='ele',columns='gmm_cluster',values='val'),square=True,cmap=ww,cbar_kws={"shrink": .5,'location':'top'},annot=False)
#plt.ylabel('$No.\;TM \;segments$',fontsize=10)
ax.set_xticklabels(['$PM$','$ER$','$Endo/Lyso$','$Golgi$','$Other$'],rotation=90,fontsize=8)
plt.xlabel('$Gmm\;cluster$',fontsize=8)
plt.ylabel('')
ax.set_yticklabels(['$Exp$','$Obs$'],fontsize=8,rotation=90)
plt.savefig('hyperLOPIT_enrich_heatmap.svg',bbox_inches='tight',transparent=True)

In [ ]:
for i in hyperLOPIT.index:
    if hyperLOPIT.loc[i,'acc'] in xx.index:
        
        hyperLOPIT.loc[i,'gmm'] = xx.loc[hyperLOPIT.loc[i,'acc'],'gmm_new']
        hyperLOPIT.loc[i,'idr'] = xx.loc[hyperLOPIT.loc[i,'acc'],'idr']

In [ ]:
d_gmm  = {0:'PM',1:'ER',2:'Endo/Lyso',3:'Golgi',4:'Other'}

In [ ]:
hyperLOPIT

In [ ]:
fig = plt.figure(figsize = ((12),3))#20,10
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
c = 0
axs = fig.subplots(nrows=1, ncols=4,gridspec_kw = {'hspace':.1,'wspace':.1})
for h in [0,1,2,3]:
    #gmmm_PM = [j]
    s = hyperLOPIT[(hyperLOPIT.mdSAM=='yes')&(hyperLOPIT.gmm.isin([c]))].groupby('fraction').median()['val']
    medians = []
    for i in s.index:
        medians.append({'fraction':i,'val':s.loc[i],'mdSAM':'+'})
    s = hyperLOPIT[(hyperLOPIT.mdSAM=='no')&(hyperLOPIT.gmm.isin([c]))].groupby('fraction').median()['val']
    for i in s.index:
        medians.append({'fraction':i,'val':s.loc[i],'mdSAM':'-'})
    #fig, ax = plt.subplots(figsize=(3*1.618,3)) 
    sns.stripplot(x='fraction',y='val',
                  data=hyperLOPIT[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm.isin([c]))],dodge=True,
                  hue='mdSAM',alpha=.1,s=3,order=fraction,hue_order=['no','yes'],palette=['gray','#C03830'],ax=axs[h])
    sns.stripplot(x='fraction',y='val',data=pd.DataFrame(medians),dodge=True,
                  hue='mdSAM',s=3,order=fraction,hue_order=['-','+'],palette=['k','k'],zorder=3,ax=axs[h])#,palette=['k','k'])
    sns.lineplot(x='fraction',y='val',estimator=np.median,
                  data=hyperLOPIT[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm.isin([c]))],
                  hue='mdSAM',hue_order=['no','yes'],palette=['gray','#C03830'],err_style='band',ax=axs[h])
    if h==0:
        axs[h].set_ylabel('Intensity',fontsize=16)
        axs[h].grid()
        axs[h].set_xticklabels(['$%s$'%i for i in fraction],rotation=90,size=8)
        axs[h].set_title('%s'%d_gmm[c],fontsize=16)
        axs[h].set_xlabel('Channel',fontsize=16)
        
    
    else:
        axs[h].set_ylabel('')
        axs[h].grid()
        axs[h].set_xticklabels([''])
        axs[h].set_yticklabels([''])
        axs[h].set_title('%s'%d_gmm[c],fontsize=16)
        axs[h].set_xlabel('')        
    axs[h].set_ylim([0,.4])
    axs[h].legend([],frameon=False)
    sam = len(set(hyperLOPIT[(hyperLOPIT.mdSAM=='yes')&(hyperLOPIT.gmm.isin([c]))].acc))
    no_sam = len(set(hyperLOPIT[(hyperLOPIT.mdSAM=='no')&(hyperLOPIT.gmm.isin([c]))].acc))
    axs[h].text(5,.32,'%d'%sam,fontsize=18,color='#C03830')
    axs[h].text(5,.25,'%d'%no_sam,fontsize=18,color='gray')
    
    c =c +1
#plt.savefig('HPA_anaysis/hyperLOPIT_gmmm_all.svg',bbox_inches='tight',transparent=True)

In [ ]:
hyperLOPIT['val_100'] = hyperLOPIT['val']*100

In [ ]:
hyperLOPIT

In [ ]:
fig = plt.figure(figsize = ((16),4))#20,10
axs = fig.subplots(nrows=1, ncols=4,gridspec_kw = {'hspace':.1,'wspace':.18})
orga = ['PM','ER','Golgi','Endo/Lyso']
for i in [0,1,3,2]:
    
    sns.boxplot(x='fraction',y='val_100',
                  data=hyperLOPIT[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm.isin([i]))],
                  hue='mdSAM',hue_order=['no','yes'],palette=['gray','#C03830'],ax=axs[i])
    
    if i==0:
        
        axs[i].set_xticklabels(['%s'%i for i in fraction],rotation=90,size=16)
        #axs[i].set_yticklabels(axs[i].get_yticklabels(),fontsize=16,rotation=0)
        axs[i].legend(title='mdSAM',title_fontsize=16,loc='upper right')
    else:
        axs[i].set_xticklabels([])
        #axs[i].set_yticklabels([])
        axs[i].set_xlabel('')
        axs[i].set_ylabel('')
        axs[i].legend([],frameon=False)
        axs[i].set_yticklabels(axs[i].get_yticklabels(),fontsize=16,rotation=0)
    axs[i].grid()
    #axs[i].grid()
    #axs[i].set_ylim([0,40])
    axs[i].set_title('%s'%orga[i],fontsize=16)
    sam = len(set(hyperLOPIT[(hyperLOPIT.mdSAM=='yes')&(hyperLOPIT.gmm.isin([i]))].acc))
    no_sam = len(set(hyperLOPIT[(hyperLOPIT.mdSAM=='no')&(hyperLOPIT.gmm.isin([i]))].acc))
    axs[i].text(1,1,'%d/%d'%(sam,no_sam),fontsize=16,color='k')
axs[0].set_yticklabels(axs[0].get_yticklabels(),fontsize=16,rotation=0)
axs[0].set_ylabel('% of total protein',fontsize=16)
axs[0].set_xlabel('Fraction',fontsize=16)
plt.savefig('HPA_anaysis/hyperLOPIT_gmmm_all_v2.svg',bbox_inches='tight',transparent=True)

In [ ]:
d

In [ ]:
fig = plt.figure(figsize = ((20),4))#20,10
axs = fig.subplots(nrows=1, ncols=5,gridspec_kw = {'hspace':.1,'wspace':.18})
for i in [0,1,2,3,4]:
    #fig, ax = plt.subplots(figsize=(4,4))

    sns.violinplot(x='fraction',y='val_100',split=True,fill=0,
                  data=hyperLOPIT[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm.isin([i]))],
                  hue='mdSAM',hue_order=['no','yes'],palette=['gray','#C03830'],ax=axs[i])
    #plt.ylim([0,40])
    axs[i].locator_params(axis='y',nbins=4)
    axs[i].set_yticklabels(axs[i].get_yticklabels(), rotation=0,fontsize=16)
    axs[i].set_xticklabels(axs[i].get_xticklabels(), rotation=90,fontsize=16)
    axs[i].legend([],frameon=False)
    axs[i].set_ylabel('')
    axs[i].set_xlabel('')
    axs[i].set_title('%s'%d_gmm[i])
    axs[i].grid()
    
plt.savefig('HPA_anaysis/hyperLOPIT_gmmm_all_v3.svg',bbox_inches='tight',transparent=True)
    

In [ ]:
sns.violinplot(x='fraction',y='val_100',split=True,fill=0,
                  data=hyperLOPIT[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm.isin([i]))],
                  hue='mdSAM',hue_order=['no','yes'],palette=['gray','#C03830'])

In [ ]:
hyperLOPIT

In [ ]:
hyperLOPIT.drop_duplicates(subset='acc')[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm.isin([4]))&(hyperLOPIT.fraction=='126')].mdSAM.value_counts()
#hyperLOPIT[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm.isin([0]))&(hyperLOPIT.fraction=='126')].drop_

In [ ]:
fig, ax = plt.subplots(figsize=(3,3*1.618)) 
cmap = matplotlib.colors.LinearSegmentedColormap.from_list("", ['#3A68AE',"#F89B35",'#D8AEC8',"#6AB6B7","#94C47D"]) #["#3A68AE",'#C2AFF0',"#09BC8A","#F28F3B","#726E60"]["#3A68AE",'#925082B2',"#434E6C"]##FF9B00","#2E4185"]
colors =  ['#194D63','#00FFFFFF','#00FFFFFF','#00FFFFFF','#00FFFFFF'] #['#3A68AE',"#F89B35",'#D8AEC8',"#6AB6B7","#94C47D"]
ax  = sns.scatterplot(x='x',y='y',data=xx[xx.gmm_new==0],hue='gmm_new',s=30,edgecolor=None,palette=colors,legend=[int(i) for i in set(xx.gmm_new)],rasterized=True)
    sns.stripplot(x='fraction',y='val',
                  data=hyperLOPIT[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm.isin([c]))],dodge=True,
                  hue='mdSAM',alpha=.1,s=3,order=fraction,hue_order=['no','yes'],palette=['gray','#C03830'],ax=axs[h])

sns.scatterplot(x='x',y='y',data=xx[(xx.mdSAM=='+')&(xx.gmm_new==0)],s=30,edgecolor='w',color='#194D63',alpha=1)##C03830

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles, [int(i) for i in range(5)],ncol=1,fontsize=7,loc='lower left')
plt.xlabel('$t‑SNE$')
ax.grid()
plt.ylabel('$t‑SNE$')

plt.ylim([-65,65])
plt.xlim([-75,75])
plt.locator_params(axis='y', nbins=4)
plt.locator_params(axis='x', nbins=4)

plt.savefig('HPA_anaysis/hyperLOPIT_tsne_v2.svg',bbox_inches='tight',transparent=True)

In [ ]:
fig, ax = plt.subplots(figsize=(2,2)) 
sns.lineplot(x='frac',y='val',
             data=frac_plot_df[(frac_plot_df.mdSAM=='-')&(frac_plot_df.gmm_new==0)],
             units='acc',estimator=None,alpha=.1,c='gray',rasterized=False)

sns.lineplot(x='frac',y='val',
             data=frac_plot_df[(frac_plot_df.mdSAM=='+')&(frac_plot_df.gmm_new==0)],
             units='acc',estimator=None,alpha=.1,c='#C03830',rasterized=False)
ax.fill_between([.5,1.5], 0,0.4 
               ,color='#18827E', alpha=1)
plt.ylim([0,.27])
plt.ylabel('$Intensity$')
plt.xlabel('')

plt.ylim([0,.27])
ax.grid()
plt.legend([],frameon=False)
plt.xticks(rotation=45)
plt.text(0.5,.03,'$n=244\,MPs$',fontsize=10)
ax.set_xticklabels([])
#plt.text(1,.35,'$Membrane$',fontsize=10)
plt.legend(['+','-'],title='$mdSAM$',fontsize=10,loc='upper right')
# plt.title('$hyperLOPIT$\n$_{U2OS\;cells}$')
plt.savefig('HPA_anaysis/hyperLOPIT_raw_PM.svg',bbox_inches='tight',transparent=True)

In [ ]:
frac_plot_df['val_100'] = 100*frac_plot_df['val'] 

In [ ]:
# fig, ax = plt.subplots(figsize=(3,3)) 
# sns.lineplot(x='frac',y='val_100',
#              data=frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.mdSAM=='-')],
#              units='acc',estimator=None,alpha=.1,c='#B1B1B2',rasterized=False)
# plt.ylim([0,31])
# plt.locator_params(axis='y',nbins=4)
# plt.xticks(fontsize=12)
# plt.yticks(fontsize=12)
# fig, ax = plt.subplots(figsize=(3,3)) 
# sns.lineplot(x='frac',y='val_100',
#              data=frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.mdSAM=='+')],
#              units='acc',estimator=None,alpha=.1,c='#C03830',rasterized=False)
# plt.ylim([0,31])
# plt.locator_params(axis='y',nbins=4)
# plt.xticks(fontsize=12)
# plt.yticks(fontsize=12)

fig, ax = plt.subplots(figsize=(3,3)) 
sns.lineplot(x='frac',y='val_100',hue='mdSAM',hue_order=['-','+'],
             data=frac_plot_df[(frac_plot_df.gmm_new==0)],
             units='acc',estimator=None,alpha=.1,palette=['#B1B1B2','#C03830'],rasterized=False)
plt.ylim([0,31])
plt.locator_params(axis='y',nbins=4)
plt.xticks(fontsize=12)
plt.xticks(fontsize=10,rotation=45)
plt.xlabel('')
plt.ylabel('')
plt.savefig('HPA_anaysis/hyperLOPIT_raw_PM.svg',bbox_inches='tight',transparent=True)

In [ ]:
fraction

In [ ]:
fig, ax = plt.subplots(figsize=(3,3)) 
sns.stripplot(x='frac',y='val_100',hue='mdSAM',hue_order=['-','+'],
             data=frac_plot_df[(frac_plot_df.gmm_new==0)],order=fraction,
             alpha=.25,palette=['#B1B1B2','#C03830'],dodge=True)
plt.ylim([0,31])
plt.locator_params(axis='y',nbins=4)
plt.yticks(fontsize=12)
plt.xticks(fontsize=10,rotation=45)
plt.xlabel('')
plt.ylabel('')
ax.fill_between([.5,1.5], 0,31 
               ,color='#F1AB3D', alpha=.51)
plt.savefig('HPA_anaysis/hyperLOPIT_raw_PM.svg',bbox_inches='tight',transparent=True)

In [ ]:
len(set(frac_plot_df[(frac_plot_df.gmm_new==0)].acc))

In [ ]:
fig, ax = plt.subplots(figsize=(3,3)) 
sns.boxplot(x='frac',y='val',
             data=frac_plot_df[frac_plot_df.frac=='127C'],hue='gmm_new')

In [ ]:
cluster = 0
l = []
for pre in [.5,0.7, 0.8]:
    temp =frac_plot_df[(frac_plot_df.gmm_new==cluster)].groupby(['mdSAM','frac']).agg(min_val = ('val','min'), percentile_ = ('val',lambda x: x.quantile(pre)))


    for i in  fraction:
        d = (temp.loc[('-',  i)].percentile_) - (temp.loc[('+',  i)].percentile_)
        l.append({'delta':d,'frac':i,'pre':pre})
fig, ax = plt.subplots(figsize=(3*1.618,3)) 
colors = cmr.take_cmap_colors(cmap, 5, return_fmt='hex')
sns.pointplot(x='frac',y='delta',data=pd.DataFrame(l),ax=ax,palette='Greys',hue='pre',join=True)
ax.axhline(0,color='k')
left, bottom, width, height = [0.58, 0.65, 0.19*1.618, 0.19]
ax2 = fig.add_axes([left, bottom, width, height])
sns.lineplot(x='frac',y='val',lw=1,ls='-',
             data=frac_plot_df[frac_plot_df.gmm_new==cluster],estimator=None,units='acc',alpha=.1
             ,c=colors[cluster],marker="o",markersize=0,markeredgecolor=None,ax=ax2)
ax2.spines[['right', 'top']].set_visible(True)
ax2.set_xticks([])
ax2.set_yticks([])
ax2.set_xlabel('')
ax2.set_ylabel('')
# ax.set_ylim([-.025,.025])
ax.set_xticklabels(['$%s$'%i for i in fraction],rotation=45,size=8)
#plt.tight_layout()
ax.grid()
ax.fill_between(range(-1,11), -0.005,0.005 
               ,color='#3A68AE', alpha=0.2)
ax.set_xlim(-.5,9.5)
L=ax.legend(title='$x^{th}$',ncol=1,loc='upper center',title_fontsize=12,fontsize=8)
L.get_texts()[0].set_text('50')
L.get_texts()[1].set_text('70')
L.get_texts()[2].set_text('80')
#L.get_texts()[3].set_text('85')
ax.locator_params(axis='y', nbins=4)
ax.set_xlabel('$Channels$')
ax.set_ylabel('$\Delta \;intensity$')

In [ ]:
cluster = 0
l = []
for pre in [.5,0.7, 0.8]:
    temp =data_gpcrs[(data_gpcrs.gmm_new==cluster)].groupby(['mdSAM','frac']).agg(min_val = ('val','min'), percentile_ = ('val',lambda x: x.quantile(pre)))


    for i in  fraction:
        d = temp.loc[('-',  i)].percentile_ - temp.loc[('+',  i)].percentile_
        l.append({'delta':d,'frac':i,'pre':pre})
fig, ax = plt.subplots(figsize=(3*1.618,3)) 
colors = cmr.take_cmap_colors(cmap, 5, return_fmt='hex')
sns.pointplot(x='frac',y='delta',data=pd.DataFrame(l),ax=ax,palette='Blues',hue='pre',join=True)
ax.axhline(0,color='k')
left, bottom, width, height = [0.58, 0.65, 0.19*1.618, 0.19]
ax2 = fig.add_axes([left, bottom, width, height])
sns.lineplot(x='frac',y='val',lw=1,ls='-',
             data=data_gpcrs[data_gpcrs.gmm_new==cluster],estimator=None,units='acc',alpha=.1
             ,c=colors[cluster],marker="o",markersize=0,markeredgecolor=None,ax=ax2)
ax2.spines[['right', 'top']].set_visible(True)
ax2.set_xticks([])
ax2.set_yticks([])
ax2.set_xlabel('')
ax2.set_ylabel('')
#ax.set_ylim([-.055,.025])
ax.set_xticklabels(['$%s$'%i for i in fraction],rotation=45,size=8)
#plt.tight_layout()
ax.grid()
ax.fill_between(range(-1,11), -0.01,0.01 
               ,color='#3A68AE', alpha=0.2)
ax.set_xlim(-.5,9.5)
L=ax.legend(title='$x^{th}$',ncol=1,loc='upper center',fontsize=8)
L.get_texts()[0].set_text('50')
L.get_texts()[1].set_text('70')
L.get_texts()[2].set_text('80')
ax.set_xlabel('$Channels$')

In [ ]:
len(set(frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.mdSAM=='+')].acc))

In [ ]:
data_gpcrs = frac_plot_df[(frac_plot_df.acc.isin(set.union(set(c_term_no_mdSAM),set(gpcr_c_term_acc))))&(frac_plot_df.gmm_new==0)]

In [ ]:
pd.Series(list(set(data_gpcrs.acc))).to_clipboard(index=0)

In [ ]:
fig, ax = plt.subplots(figsize=(2.5,2.5*1.618)) 
data = data_gpcrs[(data_gpcrs.gmm_new==0)]
sns.stripplot(x='frac',y='val',data=data_gpcrs,dodge=True,hue='mdSAM',alpha=.5,s=3,order=fraction,hue_order=['-','+'],palette=['gray','#C03830'])

sns.stripplot(x='frac',y='val',data=data_gpcrs.groupby(['frac','mdSAM']).mean(),dodge=True,hue='mdSAM',order=fraction,hue_order=['-','+'],palette=['gray','#C03830'])
ax.grid()

plt.legend([])
ax.set_xticklabels(['$%s$'%i for i in fraction],rotation=45,size=8)

In [ ]:
set(data_gpcrs[data_gpcrs.mdSAM=='+'].acc)

In [ ]:
fig, ax = plt.subplots(figsize=(2*1.618,2)) 
temp =frac_plot_df[(frac_plot_df.gmm_new==1)].groupby(['mdSAM','frac']).agg(min_val = ('val','min'), percentile_ = ('val',lambda x: x.quantile(0.75)))
l = []

for i in  fraction:
    d = temp.loc[('+',  i)].percentile_ - temp.loc[('-',  i)].percentile_
    l.append({'delta':d,'frac':i})
sns.pointplot(x='frac',y='delta',data=pd.DataFrame(l),ax=ax,color='#3A68AE')
ax.axhline(0)
left, bottom, width, height = [0.55, 0.21, 0.2*1.618, 0.2]
ax2 = fig.add_axes([left, bottom, width, height])
sns.lineplot(x='frac',y='val',lw=1,ls='-',
             data=frac_plot_df[frac_plot_df.gmm_new==1],estimator=None,units='acc',alpha=.1
             ,c='#3A68AE',marker="o",markersize=0,markeredgecolor=None,ax=ax2)
ax2.spines[['right', 'top']].set_visible(True)
ax2.set_xticks([])
ax2.set_yticks([])
ax2.set_xlabel('')
ax2.set_ylabel('')
ax.set_ylim([-.02,.015])
ax.set_xticklabels(['$%s$'%i for i in fraction],rotation=45,size=8)

In [ ]:
fig, ax = plt.subplots(figsize=(2*1.618,2)) 
ax = sns.histplot(x='val',data=frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.frac=='127C')&(frac_plot_df.mdSAM=='-')],kde=True,
             line_kws={'color': 'green', 'lw': 0, 'ls': '-'},stat='proportion',lw=0,alpha=.3,color='gray',fill=True,bins=10)
sns.histplot(x='val',data=frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.frac=='127C')&(frac_plot_df.mdSAM=='+')],
             line_kws={'color': 'green', 'lw': 0, 'ls': '-'},kde=True,stat='proportion',lw=0,alpha=.3,color='#C03830',fill=True,bins=10)
ax.lines[0].set_color('gray')
ax.lines[1].set_color('#C03830')
ax.grid()
ax2 = fig.add_axes([left, bottom, width, height])
sns.lineplot(x='frac',y='val',lw=1,ls='-',
             data=frac_plot_df[frac_plot_df.gmm_new==cluster],estimator=None,units='acc',alpha=.1
             ,c=colors[cluster],marker="o",markersize=0,markeredgecolor=None,ax=ax2)
ax2.spines[['right', 'top']].set_visible(True)
ax2.set_xticks([])
ax2.set_yticks([])
ax2.set_xlabel('')
ax2.set_ylabel('')
#plt.xlim([.11,.27])

In [ ]:
frac_plot_df = pd.read_pickle('HPA_anaysis/frac_plot_df.pkl')

In [ ]:
frac_plot_df

In [ ]:
#frac_plot_df.to_pickle('HPA_anaysis/frac_plot_df.')

In [ ]:
cluster = 0
frac = '127C'
frac_plot_df['box']='A'
frac_plot_df.loc[frac_plot_df[(frac_plot_df.gmm_new==cluster)&(frac_plot_df.frac==frac)].index,'ploting_r'] = 'main'
frac_plot_df.loc[frac_plot_df[(frac_plot_df.gmm_new==cluster)&(frac_plot_df.frac!=frac)].index,'ploting_r'] = 'back'

frac_plot_df['precent']=frac_plot_df['val']*100
fig, ax = plt.subplots(figsize=(2.4,2.4*1.618)) 
ax = sns.stripplot(x='box',y='precent',data=frac_plot_df[(frac_plot_df.gmm_new==cluster)&(frac_plot_df.frac==frac)],hue='mdSAM',dodge=True,palette=['gray','#C03830'],
                  hue_order=['-','+'],alpha=.75)
ax = sns.boxplot(x='box',y='precent',
                 data=frac_plot_df[(frac_plot_df.gmm_new==cluster)&(frac_plot_df.frac==frac)],
                 hue='mdSAM',hue_order=['-','+'],palette=['w','w'],boxprops=dict(alpha=1),showfliers = False)

# ax = sns.histplot(x='val',data=frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.frac=='127C')&(frac_plot_df.mdSAM=='-')],kde=True,
#              line_kws={'color': 'green', 'lw': 0, 'ls': '-'},stat='proportion',lw=0,alpha=.3,color='gray',fill=True,bins=10)
# ax = sns.histplot(x='val',data=frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.frac=='127C')&(frac_plot_df.mdSAM=='+')],
#              line_kws={'color': 'green', 'lw': 0, 'ls': '-'},kde=True,stat='proportion',lw=0,alpha=.3,color='#C03830',fill=True,bins=10)


left, bottom, width, height = [0.58, 0.68, 0.19*1.618, 0.19]
ax.grid()
ax2 = fig.add_axes([left, bottom, width, height])
sns.scatterplot(x='frac',y='val',
             data=frac_plot_df[(frac_plot_df.gmm_new==cluster)],hue='ploting_r',edgecolor=None,alpha=.1,palette=['#79AEA3','#3A68AE'],s=5)
ax2.spines[['right', 'top']].set_visible(True)
ax2.set_xticks([])
ax2.set_yticks([])
ax2.set_xlabel('')
ax2.set_ylabel('')
ax.legend([],frameon=False)
ax.locator_params(axis='y', nbins=6)
ax2.legend([],frameon=False)
ax.set_xlabel('')
ax.set_xticks([])
ax.set_ylabel('$\%\;(in\;fraction)$')

In [ ]:
fig, ax = plt.subplots(figsize=(2,3)) 
ax = sns.stripplot(x='box',y='precent',data=frac_plot_df[(frac_plot_df.gmm_new==cluster)&(frac_plot_df.frac==frac)],hue='mdSAM',dodge=True,palette=['gray','#C03830'],
                  hue_order=['-','+'],alpha=.75,s=4,jitter=False)
ax = sns.boxplot(x='box',y='precent',
                 data=frac_plot_df[(frac_plot_df.gmm_new==cluster)&(frac_plot_df.frac==frac)],
                 hue='mdSAM',hue_order=['-','+'],fill=False,palette=['gray','#C03830'])
plt.legend([],frameon=False)

In [ ]:
from scipy import stats
fig, ax = plt.subplots(figsize=(2.4,2.4*1.618)) 
sns.scatterplot(x='idr',y='precent',
                data=frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.frac=='127C')]
                ,hue='mdSAM',edgecolor=None,palette=['#C03830','gray'])
plt.xscale('log')
print(stats.spearmanr(xx.loc[xx[(xx.gmm_new==0)&(~pd.isna(xx.idr))].index,'idr'],
               xx.loc[xx[(xx.gmm_new==0)&(~pd.isna(xx.idr))].index,'127C']))
ax.set_ylabel('$\%\;(in\;fraction)$')
ax.set_xlabel('$No.\;reidues\;pLDDT<70$')
ax.grid()
ax.set_ylim([10,28])
ax.locator_params(axis='y', nbins=6)
plt.text(10,11,'$\\rho = 0.19$')

In [ ]:
gcpr_figure = []
for i in xx[xx.index.isin(xx[xx.index.isin(c_term_no_mdSAM)].index)].index:
    for f in fraction:
        gcpr_figure.append({'fraction':f,'val':xx.loc[i,f],'mdSAM':'-'})
for i in xx[xx.index.isin(xx[xx.index.isin(gpcr_c_term_acc)].index)].index:
    for f in fraction:
        gcpr_figure.append({'fraction':f,'val':xx.loc[i,f],'mdSAM':'+'})
gcpr_figure_mean = []
s = xx[xx.index.isin(xx[xx.index.isin(c_term_no_mdSAM)].index)].mean()[fraction]
for i in s.index:
    gcpr_figure_mean.append({'fraction':i,'val':s.loc[i],'mdSAM':'-'})
s = xx[xx.index.isin(xx[xx.index.isin(gpcr_c_term_acc)].index)].mean()[fraction]
for i in s.index:
    gcpr_figure_mean.append({'fraction':i,'val':s.loc[i],'mdSAM':'+'})
fig, ax = plt.subplots(figsize=(3,3)) 
sns.stripplot(x='fraction',y='val',data=pd.DataFrame(gcpr_figure),dodge=True,hue='mdSAM',alpha=.3,s=3,order=fraction)
sns.stripplot(x='fraction',y='val',data=pd.DataFrame(gcpr_figure_mean),dodge=True,hue='mdSAM',s=5,order=fraction)
ax.set_xticklabels(['$%s$'%i for i in fraction],rotation=45,size=8)
plt.ylabel('$Intensity$')
#ax.set_xticklabels(['$%s$'%i for i in fraction],rotation=45,size=8)

In [ ]:
def cohend(m,p):
    log_data1 = np.log(m)
    log_data2 = np.log(p)

    mean1 = np.mean(log_data1)
    mean2 = np.mean(log_data2)

    std1 = np.std(log_data1, ddof=1)
    std2 = np.std(log_data2, ddof=1)
    pooled_std = np.sqrt(((len(log_data1)-1)*std1**2 + (len(log_data2)-1)*std2**2) / (len(log_data1) + len(log_data2) - 2))
    d = (mean1 - mean2) / pooled_std
    return(d)

In [ ]:
gmmm_PM = 0
for frac in fraction:
    p = hyperLOPIT[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm==gmmm_PM)&(hyperLOPIT.mdSAM=='yes')&(hyperLOPIT.fraction==frac)].val
    m = hyperLOPIT[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm==gmmm_PM)&(hyperLOPIT.mdSAM=='no')&(hyperLOPIT.fraction==frac)].val
    print(frac)
    print(cohend(m,p))
    print(stats.ttest_ind(p,m)[-1])

In [ ]:
hyperLOPIT[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm==gmmm_PM)&(hyperLOPIT.fraction=='128C')&(hyperLOPIT.mdSAM=='no')]

In [ ]:
m = hyperLOPIT[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm.isin([0]))&(hyperLOPIT.fraction=='127C')&(hyperLOPIT.mdSAM=='no')].val.median()
p = hyperLOPIT[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm.isin([0]))&(hyperLOPIT.fraction=='127C')&(hyperLOPIT.mdSAM=='yes')].val.median()
print(m,p)

In [ ]:
sns.boxplot(x='box',y='val',data=hyperLOPIT[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm.isin([0]))&
                                            (hyperLOPIT.fraction.isin(['127C']))],hue='mdSAM',notch=True)#,common_norm=False,fill=True)

#sns.kdeplot(hyperLOPIT[(hyperLOPIT.tm=='yes')&(hyperLOPIT.gmm==5)&(hyperLOPIT.mdSAM=='no')&(hyperLOPIT.fraction=='127C')].val,color='gray')

In [ ]:
import csv

#GPCRdb = pd.read_csv('/Users/aelazar/notebooks_for_paper/GPCR_ThT/Book1.txt',delimiter='\t',quoting=csv.QUOTE_NONE, error_bad_lines=False) # https://wiki.stjude.org/display/C3D/GPCR+common+data
GPCRdb = pd.read_csv('/Users/aelazar/notebooks_for_paper/GPCR_ThT/Book1.txt',
                     delimiter='\t',
                     quoting=csv.QUOTE_NONE,
                     on_bad_lines='skip')
ind_or = []
for i in GPCRdb[~pd.isna(GPCRdb['UniProt ID'])].index:
    #print(GPCRdb.loc[i,'UniProt ID'][0:2])
    if GPCRdb.loc[i,'UniProt ID'][0:2]=='OR':
        ind_or.append(i)

In [ ]:
GPCRdb.loc[ind_or,'OR'] = 1 

In [ ]:
gpcr_acc = GPCRdb[GPCRdb.OR!=1]['#UniProt accession']
gpcr_c_term_dSAM_acc = ['O15218','O60883','P21728','P23945','P28335','P34969','P35348','P41180','P41586','P41594','P49190','P51679','P51685','Q13255','Q13585','Q5T848','Q86SP6','Q8IWK6','Q8IZP9','Q8TDV2','Q8TDV5','Q8WXD0','Q92633','Q99678','Q9BZJ6','Q9GZQ6','Q9H1Y3','Q9NSD7','Q9UBS5','Q9UPC5','Q6PRD1','O75473','O60242','Q969F8','P32247','O60353','O43603','Q6DWJ6','O94910','Q9BXB1','O14514','Q9HBX8','O60241','O00144','O95490']


In [ ]:
mdSAMs = pd.read_pickle('../files/dSAM_MPs_v_paper_v2_new_locs.pkl')

In [ ]:
gpcr_no_mdSAM = set.difference(set(gpcr_acc),set(mdSAMs.acc))
gpcr_mdSAM = set.intersection(set(gpcr_acc),set(mdSAMs.acc))
gpcr_frac = frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.frac=='127C')&(frac_plot_df.acc.isin(gpcr_acc))]

fig, ax = plt.subplots(figsize=(2,3))
sns.violinplot(fill=False,y='val_100',data=gpcr_frac,hue='mdSAM',split=True,inner=None,
              hue_order=['-','+'],palette=['gray','#C03830'])
sns.boxplot(fill=False,y='val_100',data=gpcr_frac,
            hue='mdSAM', hue_order=['-','+'],palette=['gray','#C03830'],width=.2,showcaps=False)
# sns.stripplot(y='val_100',data=frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.frac=='127C')],
#             hue='mdSAM', hue_order=['-','+'],palette=['gray','#C03830'],dodge=True,s=2)
plt.ylim([8,31])
plt.locator_params(axis='y',nbins=4)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.axhline(20,c='k',ls='--')
plt.legend([],frameon=False)
plt.ylabel('')
#plt.savefig('../frac_hyper_pm_new_mdSAM_pos_v5.svg',bbox_inches='tight',transparent=True)

In [ ]:
KeyWord_df = pd.read_csv('/Users/aelazar/Downloads/keywords_all_2023_10_27.tsv',delimiter='\t')
hTMs_uniprot = pd.read_csv('/Users/aelazar/Downloads/idmapping_2023_10_27.tsv',delimiter='\t')
hTMs_uniprot = hTMs_uniprot[~pd.isna(hTMs_uniprot.Keywords)]
z = []
for i in set(KeyWord_df.index):
    key = KeyWord_df.loc[i,'Keyword ID']
    key_temp = hTMs_uniprot[hTMs_uniprot['Keyword ID'].str.find(key)!=-1].From.values
    z.append({'name':KeyWord_df.loc[i,'Name'],'key':key,'size':len(frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.frac=='127C')&
                                        (frac_plot_df.acc.isin(key_temp))])})
zz = pd.DataFrame(z).sort_values(by='size')  
zz.iloc[-60:]

In [ ]:
name_map = {}
for i in hTMs_uniprot.index:
    name_map[hTMs_uniprot.loc[i,'Entry']] = hTMs_uniprot.loc[i,'Gene Names'].split(' ')[0]
frac_plot_df['gene_name'] = frac_plot_df.acc.map(name_map)

In [ ]:
frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.mdSAM=='+')]

In [ ]:
stats.ttest_ind(frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.mdSAM=='+')&(frac_plot_df.frac=='126')].val_100,
               frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.mdSAM=='-')&(frac_plot_df.frac=='126')].val_100)

In [ ]:
from adjustText import adjust_text


In [ ]:
for k in ['0965','0297','0829','0401','0879','0524','0770']:
    key_temp = hTMs_uniprot[hTMs_uniprot['Keyword ID'].str.find('KW-%s'%k)!=-1].From.values
    
    key_frac = frac_plot_df[(frac_plot_df.gmm_new==0)&(frac_plot_df.frac=='127C')&(frac_plot_df.acc.isin(key_temp))]
    fig, ax = plt.subplots(figsize=(2,3))
    # sns.violinplot(fill=False,y='val_100',data=key_frac,hue='mdSAM',split=True,inner=None,
    #               hue_order=['-','+'],palette=['gray','#C03830'])
    sns.boxplot(fill=False,y='val_100',data=key_frac,
                hue='mdSAM', hue_order=['-','+'],palette=['gray','#C03830'],width=.8,showcaps=False)
    sns.stripplot(y='val_100',data=key_frac,
                hue='mdSAM', hue_order=['-','+'],palette=['gray','#C03830'],dodge=True,s=5)
    # plt.ylim([8,31])
    plt.title(zz[zz.key=='KW-%s'%k].name.values[0])
    plt.locator_params(axis='y',nbins=4)
    plt.yticks(fontsize=12)
    plt.xticks(fontsize=12)
    plt.axhline(20,c='k',ls='--')
    plt.legend([],frameon=False)
    texts = []
    for i in key_frac[key_frac.mdSAM=='-'].index:
        texts.append(ax.text(-.5,key_frac.loc[i,'val_100'],
                '%s'%key_frac.loc[i,'gene_name'],
                              fontsize=8))
    # adjust_text(texts, autoalign='xy',
    #             arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)#
    
    texts = []
    for i in key_frac[key_frac.mdSAM=='+'].index:
        texts.append(ax.text(.5,key_frac.loc[i,'val_100'],
                '%s'%key_frac.loc[i,'gene_name'],
                              fontsize=8))
    # adjust_text(texts, autoalign='y',
    #             arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)#
    #             arrowprops=dict(arrowstyle="-", color='k', lw=1),zorder=0)
    plt.ylabel('')
    plt.savefig('../frac_hyper_pm_new_mdSAM_%s.svg'%k,bbox_inches='tight',transparent=True)
    print(zz[zz.key=='KW-%s'%k].name.values[0])
    print(stats.ttest_ind(key_frac[key_frac.mdSAM=='-'].val_100,
                        key_frac[key_frac.mdSAM=='+'].val_100))
    key_frac.to_pickle('frac_KW-%s.pkl'%k)

In [ ]:
frac_

In [ ]:
from matplotlib.ticker import MaxNLocator

for id in ['0965','0297','0829','0401','0879','0524','0770']:
    U2OS_ = pd.read_pickle(f'BioGrid_anaysis/U20S_KW-{id}.pkl')
    frac_ = pd.read_pickle(f'frac_KW-{id}.pkl')
    frac_['TPM_log'] = frac_.gene_name.map(dict(zip(U2OS_['Gene name'], U2OS_['TPM_log'])))

    g = sns.JointGrid(x='TPM_log', y='val_100', data=frac_, height=4, space=0)
    g.plot_joint(sns.scatterplot, hue='mdSAM', data=frac_, edgecolor=None, s=50,
                 hue_order=['-', '+'], palette=['#B1B1B2', '#C03830'])
    g.plot_marginals(sns.boxplot, hue='mdSAM', data=frac_, fill=0, showcaps=False,
                     hue_order=['-', '+'], palette=['#B1B1B2', '#C03830'])

    g.ax_joint.axhline(20, ls='--', color='k')

    # Optional: force limits to help clean ticks (uncomment if needed)
    # g.ax_joint.set_xlim(frac_['TPM_log'].min() - 0.5, frac_['TPM_log'].max() + 0.5)

    # Set exactly 4 major ticks on both axes
    g.ax_joint.xaxis.set_major_locator(MaxNLocator(nbins=4))
    g.ax_joint.yaxis.set_major_locator(MaxNLocator(nbins=4))

    # Label genes
    texts = []
    for i in frac_.index:
        texts.append(g.ax_joint.text(frac_.loc[i, 'TPM_log'], frac_.loc[i, 'val_100'],
                                     frac_.loc[i, 'gene_name'], fontsize=10))
    adjust_text(texts, autoalign='xy', ax=g.ax_joint,
                arrowprops=dict(arrowstyle="-", color='k', lw=1), zorder=0)

    # Final tick label formatting (clean numeric labels)
    g.ax_joint.set_xticklabels([f'{x:.1f}' for x in g.ax_joint.get_xticks()], fontsize=14)
    g.ax_joint.set_yticklabels([f'{y:.0f}' for y in g.ax_joint.get_yticks()], fontsize=14)

    plt.suptitle(zz[zz.key == f'KW-{id}'].name.values[0])
    plt.savefig('../U2OS_frac_Joint_pm_new_mdSAM_%s.svg'%id,bbox_inches='tight',transparent=True)


In [ ]:
for gmm in [0,1,2,3,4]:
    fig, ax = plt.subplots(figsize=(3,3)) 
    sns.stripplot(x='frac',y='val_100',hue='mdSAM',hue_order=['-','+'],order=fraction,
                 data=frac_plot_df[(frac_plot_df.gmm_new==gmm)],
                 alpha=.25,palette=['#B1B1B2','#C03830'],dodge=True)
    plt.ylim([0,35])
    plt.locator_params(axis='y',nbins=4)
    plt.yticks(fontsize=12)
    plt.xticks(fontsize=10,rotation=45)
    plt.xlabel('')
    plt.ylabel('')
    for frac in enumerate(fraction):
    
        p_val = (stats.ks_2samp(frac_plot_df[(frac_plot_df.gmm_new==gmm)&(frac_plot_df.mdSAM=='+')&(frac_plot_df.frac==frac[1])].val_100,
               frac_plot_df[(frac_plot_df.gmm_new==gmm)&(frac_plot_df.mdSAM=='-')&(frac_plot_df.frac==frac[1])].val_100)[-1])
        if p_val>.05:
            ax.text(frac[0],5,'ns',)
        else:
            ax.text(frac[0],5,'%.2f'%p_val,rotation=90)
    # ax.fill_between([.5,1.5], 0,31 
                   # ,color='#F1AB3D', alpha=.51)
        plt.savefig('HPA_anaysis/hyperLOPIT_raw_gmm_%d.svg'%gmm,bbox_inches='tight',transparent=True)

In [ ]:
p_val

In [ ]:
set(frac_plot_df.frac)

In [ ]:
gpcr_acc_or = GPCRdb[GPCRdb.OR==1]['#UniProt accession']

In [ ]:
para = pd.read_csv('panther/dSAMs_Paralogs.tsv',delimiter='\t')
para = para.drop_duplicates()

In [ ]:
l= []
for acc in tqdm.tqdm(xx[(xx.index.isin(dSAM.acc))&(xx.gmm_new==0)].index):
    if len(para[para.acc1.str.find(acc)!=-1])>=1:
        if xx[xx.index.isin([acc])].empty ==False:
            for frac in set(hyperLOPIT.fraction):
                reff = xx[xx.index==acc][frac].values[0]
                if xx[(xx.index.isin([i.split('=')[-1] for i in para[para.acc1.str.find(acc)!=-1].acc2]))].empty ==False:
                    for paralog in xx[(xx.index.isin([i.split('=')[-1] for i in para[para.acc1.str.find(acc)!=-1].acc2]))].index:
                        if paralog not in set(dSAM.acc):
                            delta = delta = (xx[xx.index==paralog][frac].values[0]-reff)
                            l.append({'acc':acc,'paralog':paralog,'frac':frac,'delta':delta})#(acc,paralog,delta)
 #               print(acc,len(xx[xx.index.isin([i.split('=')[-1] for i in para[para.acc1.str.find(acc)!=-1].acc2])]))
                
ll= []
for acc in tqdm.tqdm(xx[(~xx.index.isin(dSAM.acc))&(xx.gmm_new==0)].acc):
    if len(para[para.acc1.str.find(acc)!=-1])>=1:
        if xx[xx.index.isin([acc])].empty ==False:
            for frac in set(hyperLOPIT.fraction):
                reff = xx[xx.index==acc][frac].values[0]
                if xx[(xx.index.isin([i.split('=')[-1] for i in para[para.acc1.str.find(acc)!=-1].acc2]))].empty ==False:
                    for paralog in xx[(xx.index.isin([i.split('=')[-1] for i in para[para.acc1.str.find(acc)!=-1].acc2]))].index:
                        #if paralog not in set(dSAM.acc):
                        delta = (xx[xx.index==paralog][frac].values[0]-reff)
                        ll.append({'acc':acc,'paralog':paralog,'frac':frac,'delta':delta})#(acc,paralog,delta)
 #               print(acc,len(xx[xx.index.isin([i.split('=')[-1] for i in para[para.acc1.str.find(acc)!=-1].acc2])]))
                


In [ ]:
pd.Series(xx[xx.gmm_new==0].index).to_clipboard(index=0)

In [ ]:
delta = pd.DataFrame(ll)   
#sns.stripplot(y='delta',data=delta,x='frac',order=fraction)
fig, ax = plt.subplots(figsize=(4,4))
sns.pointplot(y='delta',data=pd.DataFrame(l) ,x='frac',order=fraction,errorbar='se',color='r',estimator=np.median)
sns.pointplot(y='delta',data=delta,x='frac',order=fraction,errorbar='se',estimator=np.median)
#sns.kdeplot(x='delta',data=delta[delta.frac=='131'])
plt.axhline(0)

In [ ]:
sns.boxplot(y='delta',data=pd.DataFrame(l) ,x='frac',order=fraction)

In [ ]:
sns.ecdfplot(x='delta',data=pd.DataFrame(l),hue='frac')#,order=fraction)
#sns.ecdfplot(x='delta',data=pd.DataFrame(ll),hue='frac')
#plt.axvline(0)

In [ ]:
stats.ks_2samp(delta[delta.frac=='127N'].delta,
              delta[delta.frac=='130C'].delta)

In [ ]:
sns.scatterplot(x='x',y='y',data = xx,s=10,c='k',edgecolor=None)#P02786

sns.scatterplot(x='x',y='y',data=xx[xx.acc.isin(accs)],c='k',hue='mdSAM',s=10,edgecolor=None)#P02786
#ns.scatterplot(x='x',y='y',data=xx,c='k',hue='k-means',s=20,edgecolor=None)#P02786

In [ ]:

sns.scatterplot(x='x',y='y',data=xx[xx.Compartment.isin(['ER','Endosome','Golgi','Plasma membrane','Lysosome'])],s=30,hue='Compartment',edgecolor=None)#P02786
sns.scatterplot(x='x',y='y',data=xx[(xx.acc.isin(accs))],c='r',s=20,hue='mdSAM',edgecolor=None,palette=['r','k'])#P02786

In [ ]:
fig, ax = plt.subplots(figsize=(6,3)) 
sns.lineplot(x='fraction',y='val',data=hyperLOPIT[(hyperLOPIT.Compartment.isin(['ER','Endosome','Golgi','Plasma membrane','Lysosome']))],hue='Compartment')
ax.grid()
plt.xticks(rotation=90)

In [ ]:
sns.boxplot(y='delta',data=delta,x='frac',order=fraction)
#sns.kdeplot(x='delta',data=delta[delta.frac=='131'])
plt.axhline(0)

In [ ]:
print(xx.loc['Q9HBR0',['x','y']])
print(xx.loc['Q96QD8',['x','y']])

In [ ]:
fig, ax = plt.subplots(figsize=(6,6)) 
#sns.scatterplot(x='x',y='y',data=xx[xx.acc.isin(accs)])
for pair in delta[delta.acc.isin(z)].drop_duplicates(subset=['acc','paralog']).index:
    acc1 = delta.loc[pair,'acc']
    acc2 = delta.loc[pair,'paralog']
    x1 = xx.loc[acc1,'x'] 
    x2 = xx.loc[acc2,'x'] 
    y1 = xx.loc[acc1,'y'] 
    y2 = xx.loc[acc2,'y']
    # print(xx.loc[acc1,['x','y']])
    # print(x1,y1)
    # print(xx.loc[acc2,['x','y']])
    # print(x2,y2)
    # print([x1,x2],[y1,y2])
    sns.scatterplot(x='x',y='y',data=xx.loc[[acc1,acc2]],hue='mdSAM',palette=['r','k'],alpha=1,s=100)
    sns.lineplot(x=[x1,x2], y=[y1,y2],color='k',lw=1,zorder=0)
    plt.legend([])

In [ ]:
w = delta.drop_duplicates(subset=['acc','paralog'])
import math

In [ ]:
set(z)

In [ ]:
l = []
z = []
for acc in  w.acc:
    if len(w[w.acc==acc].paralog)>=1:
        z.append(acc)
        for para in w[w.acc==acc].paralog:
            #print('mdSAM-para')
            l.append({'ele':'mdSAM-para','dist':math.dist(xx.loc[acc,['x','y']],xx.loc[para,['x','y']]),'acc':acc})
        for i in w[w.acc==acc].paralog:
            for j in w[w.acc==acc].paralog:
                if i!=j:
                    l.append({'ele':'para-para','dist':math.dist(xx.loc[i,['x','y']],xx.loc[j,['x','y']]),'acc':acc})
    # print(math.dist(xx.loc['P55011',['x','y']],xx.loc['Q9UP95',['x','y']]))
# print(math.dist(xx.loc['Q9BXP2',['x','y']],xx.loc['Q9UP95',['x','y']]))


In [ ]:
ll = pd.DataFrame(l)
ll['box']='A'
sns.boxplot(x='ele',y='dist',data=ll,hue='ele')
plt.yscale('log')

In [ ]:
import numpy as np 

In [ ]:
plt.plot(np.linspace(0, 1, 1001),stats.gaussian_kde(dis_p).evaluate( np.linspace(0, 1, 1001)))
plt.plot(np.linspace(0, 1, 1001),stats.gaussian_kde(dis_m).evaluate( np.linspace(0, 1, 1001)))
plt.xlim([0.1,.3])

In [ ]:
stats.gaussian_kde(dis_m).evaluate( np.linspace(0.2, .3, 1001)).sum()

In [ ]:
stats.gaussian_kde(dis_p).evaluate( np.linspace(0.2, .3, 1001)).sum()

In [ ]:
sns.kdeplot(x='val',data=frac_plot_df[(frac_plot_df.gmm_new==0)&
                                      (frac_plot_df.frac=='127C')]
            ,hue='mdSAM',cumulative=True, common_norm=False, common_grid=True,) 
#    multiple="stack"